In [6]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
cd /content/drive/MyDrive

/content/drive/MyDrive


In [12]:
ls

'Colab Notebooks'/   KDD2022@   __MACOSX/   taxi_drop/   taxi_drop.zip


# Data Preprocess & DataLoader

In [7]:
import numpy as np
import os
import torch
from torch.utils.data import TensorDataset, DataLoader

"""数据加载和预处理"""
# data["xxx_catagory"] = (num_samples, num_time_steps, locations, channels)
# eg: data["x_train"].shape: (2606, 12, 266, 3), data["y_train"].shape: (2606, 12, 266, 1)
# 论文： 输入：（P，N，C） P：历史时间步数，N：位置数量，C：通道数量
# channel： 0: traffic flow, 1: time 0-47/48 48個 , 2: day 0-6七天
# 使用dataloader直接加载数据，和st-llm不同，没有对最后剩余凑不满的数据进行填充，因此大小会不一样，最后一个batch会变小

# 数据标准化
class StandardScaler:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def transform(self, data): #标准化数据
        return (data - self.mean) / self.std

    def inverse_transform(self, data): #还原数据
        return (data * self.std) + self.mean

def load_dataset(dataset_dir, batch_size, valid_batch_size=None, test_batch_size=None):
    data = {}
    for category in ["train", "val", "test"]:
        cat_data = np.load(os.path.join(dataset_dir, category + ".npz"))
        data["x_" + category] = cat_data["x"]
        data["y_" + category] = cat_data["y"]
    #对第一维channel进行标准化
    scaler = StandardScaler(
        mean=data["x_train"][..., 0].mean(), std=data["x_train"][..., 0].std() # 为什么：only scale the first channel traffic flow
    )
    for category in ["train", "val", "test"]:
        data["x_" + category][..., 0] = scaler.transform(data["x_" + category][..., 0])

    train_ds = TensorDataset(torch.Tensor(data["x_train"]), torch.Tensor(data["y_train"]))
    val_ds   = TensorDataset(torch.Tensor(data["x_val"]),   torch.Tensor(data["y_val"]))

    data["train_loader"] = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    data["val_loader"] = DataLoader(val_ds, batch_size=valid_batch_size, shuffle=False)

    # test_ds  = TensorDataset(torch.Tensor(data["x_test"]),  torch.Tensor(data["y_test"]))
    # data["test_loader"] = DataLoader(test_ds, batch_size=test_batch_size, shuffle=False)
    
    data["scaler"] = scaler

    return data

In [13]:
"""测试dataLoader模块"""
dataset_dir = "taxi_drop"
taxi_drop_data = load_dataset(dataset_dir, batch_size=32, valid_batch_size=32, test_batch_size=32)

In [14]:
example_x, example_y = next(iter(taxi_drop_data["train_loader"]))
print("Example x shape:", example_x.shape)  # (batch_size, num_time_steps, locations, channels)
print("Example y shape:", example_y.shape)  # (batch_size, num_time_steps, locations, 1)

Example x shape: torch.Size([32, 12, 266, 3])
Example y shape: torch.Size([32, 12, 266, 1])


In [ ]:
# example_x[0,0,100,:]

tensor([-0.6581,  0.1042,  3.0000])

# ST_LLM Model

In [15]:
import torch
import torch.nn as nn
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

In [16]:
"""時間embedding:包含一天中的时间段和一周中的星期几信息"""
#輸入： x : (B, P, N, C) B: batch_size, P：历史时间步数，N：位置数量，C：通道数量
#輸出： time_embedding shape: (B, n_embd, N, 1)
class TemporalEmbedding(nn.Module):
    def __init__(self, time, n_embd):
        super().__init__()
        self.day = time
        self.time_day = nn.Embedding(time, n_embd)
        self.time_week = nn.Embedding(7, n_embd)
        nn.init.xavier_uniform_(self.time_day.weight)
        nn.init.xavier_uniform_(self.time_week.weight)

    def forward(self, x): # x shape: (B, P, N, C)
        day_info = x[..., 1]  # (B, P, N)
        day_idx = (day_info[:, -1, :] * self.day).long().clamp(0, self.day - 1)  # (B, N)
        day_embd = self.time_day(day_idx)                      # (B, N, n_embd)
        day_embd = day_embd.transpose(1, 2).unsqueeze(-1)      # (B, n_embd, N, 1)

        week_info = x[..., 2]  # (B, P, N)
        week_idx = week_info[:, -1, :].long().clamp(0, 6)      # (B, N)
        week_embd = self.time_week(week_idx)                   # (B, N, n_embd)
        week_embd = week_embd.transpose(1, 2).unsqueeze(-1)    # (B, n_embd, N, 1)

        return day_embd + week_embd  # (B, n_embd, N, 1)


In [17]:
"""空間 embedding：每個地點學習一個獨立的向量"""
# 輸入： x : (B, P, N, C)
# 輸出： (B, n_embd, N, 1)
class SpatialEmbedding(nn.Module):
    def __init__(self, num_locations, n_embd):
        super().__init__()
        self.num_locations = num_locations
        self.location_embd = nn.Embedding(num_locations, n_embd)  # table shape: (N, n_embd)
        nn.init.xavier_uniform_(self.location_embd.weight)

    def forward(self,x):  # x shape: (B, P, N, C)
        B= x.shape[0]
        node_emb = self.location_embd(torch.arange(self.num_locations))     # (N, n_embd)
        node_emb = node_emb.unsqueeze(0).expand(B, -1, -1).transpose(1,2).unsqueeze(-1)  # (B, n_embd, N, 1)

        return node_emb  # (B, n_embd, N, 1)

In [18]:
"""信息embedding：利用卷積融合流量+時間+空間信息"""
class TokenEmbedding(nn.Module):
    def __init__(self, input_dim = 3, input_len = 12, n_embd = 256):
        super().__init__()
        #卷積：（B，256，H，W)
        self.in_channels = input_dim * input_len # 3个通道，每个通道12个时间步，展开后是36维
        self.tk_embd = nn.Conv2d(in_channels=self.in_channels, out_channels=n_embd, kernel_size=1) #卷积核大小为1
        

    def forward(self, x): # x shape: (B, P, N, C)
        B, P, N, C = x.shape
        x = x.permute(0, 3, 1, 2) # (B, C, P, N)
        x = x.reshape(B, -1, N)    # (B, C*P, N)
        tk_embd = self.tk_embd(x.unsqueeze(-1)) # (B, n_embd, N, 1)

        return tk_embd # (B, n_embd, N, 1)

In [19]:
'''测试embedding模块'''
# # random_data = example_x # (batch_size, num_time_steps, locations, channels)
# token_embd = TokenEmbedding(input_dim=3, input_len=12, n_embd=256)
# temporal_embd = TemporalEmbedding(time=48, n_embd=256)
# spatial_embd = SpatialEmbedding(num_locations=266, n_embd=256)
# tk_embd = token_embd(example_x)
# print("Token Embedding shape:", tk_embd.shape)  
# sp_embd = spatial_embd(example_x)
# print("Spatial Embedding shape", sp_embd.shape)  # (B, n_embd, N, 1)
# tp_embd = temporal_embd(example_x)
# print("Temporal Embedding shape:", tp_embd.shape)  # (B, n_embd, N, 1)
# embd_feature = torch.cat([tk_embd,tp_embd,sp_embd], dim=1) # (B, 3*n_embd, N, 1)
# print(embd_feature.shape)
# fusion = nn.Conv2d(768,768,kernel_size=(1,1))
# data_feature = fusion(embd_feature)
# print(data_feature.shape)

'测试embedding模块'

In [20]:
'''部分冻结的大语言模型'''
#GPT2Block包含：
# - ln_1: LayerNorm
# - attn: GPT2Attention
# - ln_2: LayerNorm
# - mlp: GPT2MLP
# 原始一共12层
# GPT2輸入： batch, sequence_length, embedding_dim
# TODO:這個地方用序列建模合理嗎？
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

class PFA(nn.Module):
    def __init__(self, gpt_layers = 6, U = 1): 
        super().__init__()
        self.gpt2 = GPT2Model.from_pretrained("gpt2")
        #保留中間的attention權重：self.gpt2 = GPT2Model.from_pretrained("gpt2", output_attentions=True, output_hidden_states=True）
        self.gpt2.h = self.gpt2.h[:gpt_layers] #保留前gpt_layers层，注意原始的gpt2是12層
        self.U = U #每个位置的邻居数量

        # frooze 和 learning 的 param调整
        for layer_idx, layer in enumerate(self.gpt2.h):
            for name, param in layer.named_parameters():
                if layer_idx < gpt_layers - self.U:  #所有層數 - 不凍結attention的層數,前幾層
                    #TODO: wpe是什么
                    if "ln" in name: # LayerNorm层
                        param.requires_grad = True
                    else:
                        param.requires_grad = False
                else: #最後U層，attn也要打開
                    if "mlp" in name:
                        param.requires_grad = False
                    else:
                        param.requires_grad = True

    def forward(self, x): # x shape: (batch_size, num_time_steps, locations, n_embd)
        return self.gpt2(inputs_embeds = x).last_hidden_state

In [21]:
"""測繪pfa模塊"""
# in_data = torch.randn(32, 266, 768) # (B, N, gpt_embd)
# pfa = PFA(gpt_layers=6, U=1)
# out_data = pfa(in_data)
# print("PFA output shape:", out_data.shape)  # (B, N, gpt_embd)

'測繪pfa模塊'

In [22]:
class ST_LLM(nn.Module):
    def __init__(self, batch_size = 32, num_nodes = 266, input_len = 12, output_len = 12, gpt_layers = 6, U = 1):
        super().__init__()
        self.batch_size = batch_size
        self.num_nodes = num_nodes #location数量
        self.input_len = input_len
        self.output_len = output_len
        self.gpt_layers = gpt_layers
        self.U = U
        self.time = 48      #一天中的时间段数量
        self.n_embd = 256   #时间，空间，token三个拼一起，3*256=768
        self.gpt_embd = 768 #embedding维度，和GPT-2的默认维度一致
        #embedding
        self.time_embd = TemporalEmbedding(self.time, self.n_embd) #时间编码模块
        self.spatial_embd = SpatialEmbedding(self.num_nodes, self.n_embd) #空间编码模块
        self.token_embd = TokenEmbedding(input_dim=3, input_len=self.input_len, n_embd=self.n_embd) #token embedding, 将原始的3个通道映射到n_embd维度
        #聚合embedding
        self.feature_fuse = nn.Conv2d(in_channels=3*self.n_embd, out_channels=self.gpt_embd, kernel_size=1) #特征融合，卷积核大小为1，输入（B，xxx，H，W）
        #PFA
        self.pfa = PFA(gpt_layers=self.gpt_layers, U=self.U)
        #最后解码成预测结果
        self.regression_layer = nn.Conv2d(in_channels=self.gpt_embd, out_channels=self.output_len, kernel_size=1) #卷积核大小为1，输入（B，gpt_embd，H，W）
        

    def forward(self, history_data):    # x shape: (B, P, N, C)
        input_data = history_data
        B, P, N, C = input_data.shape

        #embedding
        time_embd = self.time_embd(input_data) # (B, n_embd, N, 1)
        spatial_embd = self.spatial_embd(input_data) # (B, n_embd, N, 1)
        token_embd = self.token_embd(input_data) # (B, n_embd, N, 1)

        #feature fusing
        embd_feature = torch.cat([token_embd, time_embd, spatial_embd], dim=1) # (B, 3*n_embd, N, 1)
        data_feature = self.feature_fuse(embd_feature) # (B, gpt_embd,N,1)
        data_feature = data_feature.squeeze(-1).permute(0, 2, 1) # (B, N, gpt_embd)

        #PFA
        pfa_out = self.pfa(data_feature) # (B, N, gpt_embd)

        #最終預測
        pfa_out = pfa_out.permute(0, 2, 1).unsqueeze(-1) # (B, gpt_embd, N, 1)

        pred = self.regression_layer(pfa_out) # (B, output_len, N, 1)
        pred = pred.squeeze(-1) # (B, output_len, N)
        
        return pred

In [23]:
"""测试ST_LLM模型"""
model = ST_LLM(batch_size=32, num_nodes=266, input_len=12, output_len=12, gpt_layers=6, U=1)
pred = model(example_x)
print("Prediction shape:", pred.shape)  # (B, output_len, N, 1)
scaler = taxi_drop_data['scaler']
prediction = scaler.inverse_transform(pred[...,0]) #还原到原始尺度
print("Inverse transformed prediction shape:", prediction.shape)  # (B, output_len, N)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prediction shape: torch.Size([32, 12, 266])
Inverse transformed prediction shape: torch.Size([32, 12])


# Train

In [ ]:
# train 
import torch
import numpy as np
import pandas as pd
import argparse
import time
import util

torch.manual_seed(313)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
print("Using Device:{device}")

#TODO:自动化参数输入
lrate = 1e-3
wdecay = 1e-4
epochs = 300
num_nodes = 266
input_len = 12
output_len = 12
gpt_layers = 6
U = 1

"""训练循环和评估指标计算"""
class trainer:
    def __init__(self,scaler, learning_rate, wdecay,
                 num_nodes, input_len, output_len, gpt_layers, U):
        # scaler 是 dataloader["scaler"]
        self.scaler = scaler
        self.model = ST_LLM(batch_size=32, num_nodes=num_nodes, 
                            input_len=input_len, output_len=output_len,
                            gpt_layers=gpt_layers, U=U).to(device) #model to device
        self.optimizer = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, self.model.parameters()), 
                lr=learning_rate, 
                weight_decay=wdecay)
        self.criterion = nn.MSELoss()

    def train_loop(self, x, y):# x:(B, P, N, C), y:(B, P, N, 1)
        model = self.model
        model.train()
        self.optimizer.zero_grad()
        output = model(x).squeeze(-1)   #output: (B, output_len, N, 1)
        return
    
def main():
    # 这里可以设置训练参数，加载数据，创建trainer实例，并调用train_loop进行训练
    #TODO:参数输入
    dataset_dir = "taxi_drop"
    dataloader = load_dataset(dataset_dir, batch_size=32, valid_batch_size=32, test_batch_size=32)
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        for x_batch, y_batch in dataloader["train_loader"]:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            trainer.train_loop(x_batch, y_batch)
    pass

In [ ]:
class trainer:
    def __init__(
        self,scaler,lrate,wdecay,input_dim,num_nodes,input_len,output_len,llm_layer, U,device):
        self.model = ST_LLM(
            input_dim, num_nodes, input_len, output_len, llm_layer, U, device
        )
        self.model.to(device)
        
        self.optimizer = Ranger(self.model.parameters(), lr=lrate, weight_decay=wdecay)
        # self.optimizer = optim.Adam(self.model.parameters(), lr=lrate, weight_decay=wdecay)
        
        self.loss = util.MAE_torch
        self.scaler = scaler
        self.clip = 5
        print("The number of parameters: {}".format(self.model.param_num()))
        print(self.model)

    def train(self, input, real_val):
        self.model.train()
        self.optimizer.zero_grad()
        output = self.model(input)
        output = output.transpose(1, 3)
        real = torch.unsqueeze(real_val, dim=1)
        predict = self.scaler.inverse_transform(output)
        loss = self.loss(predict, real, 0.0)
        loss.backward()
        if self.clip is not None:
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.clip)
        self.optimizer.step()
        mape = util.MAPE_torch(predict, real, 0.0).item()
        rmse = util.RMSE_torch(predict, real, 0.0).item()
        wmape = util.WMAPE_torch(predict, real, 0.0).item()
        return loss.item(), mape, rmse, wmape

    def eval(self, input, real_val):
        self.model.eval()
        output = self.model(input)
        output = output.transpose(1, 3)
        real = torch.unsqueeze(real_val, dim=1)
        predict = self.scaler.inverse_transform(output)
        loss = self.loss(predict, real, 0.0)
        mape = util.MAPE_torch(predict, real, 0.0).item()
        rmse = util.RMSE_torch(predict, real, 0.0).item()
        wmape = util.WMAPE_torch(predict, real, 0.0).item()
        return loss.item(), mape, rmse, wmape





In [ ]:
def main():
    seed_it(6666)
    data = args.data
  
    
    device = torch.device(args.device)
    dataloader = util.load_dataset(
        args.data, args.batch_size, args.batch_size, args.batch_size
    )
    scaler = dataloader["scaler"]

    loss = 9999999
    test_log = 999999
    epochs_since_best_mae = 0
    path = args.save + data + "/"

    his_loss = []
    val_time = []
    train_time = []
    result = []
    test_result = []
    print(args)

    if not os.path.exists(path):
        os.makedirs(path)

    engine = trainer(
        scaler,
        args.lrate,
        args.wdecay,
        args.input_dim,
        args.num_nodes,
        args.input_len,
        args.output_len,
        args.llm_layer,
        args.U,
        device
        )

    print("start training...", flush=True)
    for i in range(1, args.epochs + 1):
        train_loss = []
        train_mape = []
        train_rmse = []
        train_wmape = []

        t1 = time.time()
        # dataloader['train_loader'].shuffle()
        for iter, (x, y) in enumerate(dataloader["train_loader"].get_iterator()):
            trainx = torch.Tensor(x).to(device)  # 64 12 250 1
            trainx = trainx.transpose(1, 3)
            trainy = torch.Tensor(y).to(device)
            trainy = trainy.transpose(1, 3)
            metrics = engine.train(trainx, trainy[:, 0, :, :])
            train_loss.append(metrics[0])
            train_mape.append(metrics[1])
            train_rmse.append(metrics[2])
            train_wmape.append(metrics[3])

            # if iter % args.print_every == 0:
                # log = "Iter: {:03d}, Train Loss: {:.4f}, Train RMSE: {:.4f}, Train MAPE: {:.4f}, Train WMAPE: {:.4f}"
                # print(
                #     log.format(
                #         iter,
                #         train_loss[-1],
                #         train_rmse[-1],
                #         train_mape[-1],
                #         train_wmape[-1],
                #     ),
                #     flush=True,
                # )
        t2 = time.time()
        log = "Epoch: {:03d}, Training Time: {:.4f} secs"
        print(log.format(i, (t2 - t1)))
        train_time.append(t2 - t1)

        # validation
        valid_loss = []
        valid_mape = []
        valid_wmape = []
        valid_rmse = []

        s1 = time.time()
        for iter, (x, y) in enumerate(dataloader["val_loader"].get_iterator()):
            testx = torch.Tensor(x).to(device)
            testx = testx.transpose(1, 3)
            testy = torch.Tensor(y).to(device)
            testy = testy.transpose(1, 3)
            metrics = engine.eval(testx, testy[:, 0, :, :])
            valid_loss.append(metrics[0])
            valid_mape.append(metrics[1])
            valid_rmse.append(metrics[2])
            valid_wmape.append(metrics[3])

        s2 = time.time()

        log = "Epoch: {:03d}, Inference Time: {:.4f} secs"
        print(log.format(i, (s2 - s1)))
        val_time.append(s2 - s1)

        mtrain_loss = np.mean(train_loss)
        mtrain_mape = np.mean(train_mape)
        mtrain_wmape = np.mean(train_wmape)
        mtrain_rmse = np.mean(train_rmse)

        mvalid_loss = np.mean(valid_loss)
        mvalid_mape = np.mean(valid_mape)
        mvalid_wmape = np.mean(valid_wmape)
        mvalid_rmse = np.mean(valid_rmse)

        his_loss.append(mvalid_loss)
        print("-----------------------")

        train_m = dict(
            train_loss=np.mean(train_loss),
            train_rmse=np.mean(train_rmse),
            train_mape=np.mean(train_mape),
            train_wmape=np.mean(train_wmape),
            valid_loss=np.mean(valid_loss),
            valid_rmse=np.mean(valid_rmse),
            valid_mape=np.mean(valid_mape),
            valid_wmape=np.mean(valid_wmape),
        )
        train_m = pd.Series(train_m)
        result.append(train_m)

        log = "Epoch: {:03d}, Train Loss: {:.4f}, Train RMSE: {:.4f}, Train MAPE: {:.4f}, Train WMAPE: {:.4f}, "
        print(
            log.format(i, mtrain_loss, mtrain_rmse, mtrain_mape, mtrain_wmape),
            flush=True,
        )
        log = "Epoch: {:03d}, Valid Loss: {:.4f}, Valid RMSE: {:.4f}, Valid MAPE: {:.4f}, Valid WMAPE: {:.4f}"
        print(
            log.format(i, mvalid_loss, mvalid_rmse, mvalid_mape, mvalid_wmape),
            flush=True,
        )

        if mvalid_loss < loss:
            print("###Update tasks appear###")
            if i <= 100:
                # It is not necessary to print the results of the test set when epoch is less than 100, because the model has not yet converged.
                loss = mvalid_loss
                torch.save(engine.model.state_dict(), path + "best_model.pth")
                bestid = i
                epochs_since_best_mae = 0
                print("Updating! Valid Loss:{:.4f}".format(mvalid_loss), end=", ")
                print("epoch: ", i)
                
            else: 
                loss = mvalid_loss
                torch.save(engine.model.state_dict(), path + "best_model.pth")
                bestid = i
                epochs_since_best_mae = 0
                print("Updating! Valid Loss:{:.4f}".format(mvalid_loss), end=", ")
                print("epoch: ", i)

                outputs = []
                realy = torch.Tensor(dataloader["y_test"]).to(device)
                realy = realy.transpose(1, 3)[:, 0, :, :]

                for iter, (x, y) in enumerate(dataloader["test_loader"].get_iterator()):
                    testx = torch.Tensor(x).to(device)
                    testx = testx.transpose(1, 3)
                    with torch.no_grad():
                        preds = engine.model(testx).transpose(1, 3)
                    outputs.append(preds.squeeze())

                yhat = torch.cat(outputs, dim=0)
                yhat = yhat[: realy.size(0), ...]

                amae = []
                amape = []
                awmape = []
                armse = []

                for j in range(args.output_len):
                    pred = scaler.inverse_transform(yhat[:, :, j])
                    real = realy[:, :, j]
                    metrics = util.metric(pred, real)
                    # log = "Evaluate best model on test data for horizon {:d}, Test MAE: {:.4f}, Test RMSE: {:.4f}, Test MAPE: {:.4f}, Test WMAPE: {:.4f}"
                    # print(
                    #     log.format(
                    #         j + 1, metrics[0], metrics[2], metrics[1], metrics[3]
                    #     )
                    # )

                    amae.append(metrics[0])
                    amape.append(metrics[1])
                    armse.append(metrics[2])
                    awmape.append(metrics[3])

                log = "On average over 12 horizons, Test MAE: {:.4f}, Test RMSE: {:.4f}, Test MAPE: {:.4f}, Test WMAPE: {:.4f}"
                print(
                    log.format(
                        np.mean(amae), np.mean(armse), np.mean(amape), np.mean(awmape)
                    )
                )

                if np.mean(amae) < test_log:
                    test_log = np.mean(amae)
                    print(f"Test low! Updating! Test Loss: {test_log:.4f}, Valid Loss: {mvalid_loss:.4f}, epoch: {i}")
                else:
                    epochs_since_best_mae += 1
                    print("No update")

        else:
            epochs_since_best_mae += 1
            print("No update")

        train_csv = pd.DataFrame(result)
        train_csv.round(8).to_csv(f"{path}/train.csv")

        # Early stop
        if epochs_since_best_mae >= args.es_patience and i >= 200:
            break

    # Output consumption
    print("Average Training Time: {:.4f} secs/epoch".format(np.mean(train_time)))
    print("Average Inference Time: {:.4f} secs".format(np.mean(val_time)))

    # test
    print("Training ends")
    print("The epoch of the best result：", bestid)
    print("The valid loss of the best model", str(round(his_loss[bestid - 1], 4)))

    engine.model.load_state_dict(torch.load(path + "best_model.pth"))
    outputs = []
    realy = torch.Tensor(dataloader["y_test"]).to(device)
    realy = realy.transpose(1, 3)[:, 0, :, :]

    for iter, (x, y) in enumerate(dataloader["test_loader"].get_iterator()):
        testx = torch.Tensor(x).to(device)
        testx = testx.transpose(1, 3)
        with torch.no_grad():
            preds = engine.model(testx).transpose(1, 3)
        outputs.append(preds.squeeze())

    yhat = torch.cat(outputs, dim=0)
    yhat = yhat[: realy.size(0), ...]

    amae = []
    amape = []
    armse = []
    awmape = []

    test_m = []

    for i in range(args.output_len):
        pred = scaler.inverse_transform(yhat[:, :, i])
        real = realy[:, :, i]
        metrics = util.metric(pred, real)
        # log = "Evaluate best model on test data for horizon {:d}, Test MAE: {:.4f}, Test RMSE: {:.4f}, Test MAPE: {:.4f}, Test WMAPE: {:.4f}"
        # print(log.format(i + 1, metrics[0], metrics[2], metrics[1], metrics[3]))

        test_m = dict(
            test_loss=np.mean(metrics[0]),
            test_rmse=np.mean(metrics[2]),
            test_mape=np.mean(metrics[1]),
            test_wmape=np.mean(metrics[3]),
        )
        test_m = pd.Series(test_m)
        test_result.append(test_m)

        amae.append(metrics[0])
        amape.append(metrics[1])
        armse.append(metrics[2])
        awmape.append(metrics[3])

    log = "On average over 12 horizons, Test MAE: {:.4f}, Test RMSE: {:.4f}, Test MAPE: {:.4f}, Test WMAPE: {:.4f}"
    print(log.format(np.mean(amae), np.mean(armse), np.mean(amape), np.mean(awmape)))

    test_m = dict(
        test_loss=np.mean(amae),
        test_rmse=np.mean(armse),
        test_mape=np.mean(amape),
        test_wmape=np.mean(awmape),
    )
    test_m = pd.Series(test_m)
    test_result.append(test_m)

    test_csv = pd.DataFrame(test_result)
    test_csv.round(8).to_csv(f"{path}/test.csv")
